# Modelado Predictivo: Clasificación con Random Forest
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI)

**Contexto:** Tomando como base el dataset limpio, libre de fugas de datos y estandarizado (`CTG_clean.csv`), este notebook implementa y entrena un modelo de **Random Forest** (Bosque Aleatorio). El propósito es aprovechar las 21 variables para clasificar de forma robusta el estado de salud del fetal (`NSP`), reduciendo la varianza y mitigando el sobreajuste (*overfitting*) propio de los árboles de decisión individuales mediante la combinación de *Bagging* (muestreo con reemplazo) y la decorrelación por submuestreo de características (*feature subsampling*).

### Objetivos del Modelado

1. **Construcción y Entrenamiento del Ensamble:** Ensamblar un clasificador `RandomForestClassifier` basado en múltiples árboles de decisión CART, configurando hiperparámetros clave como el número de árboles (`n_estimators`), la profundidad máxima (`max_depth`) y el límite de características por división (`max_features`).

2. **Validación Externa e Interna (OOB Score):** Implementar la validación interna mediante la puntuación *Out-Of-Bag* (`oob_score`), aprovechando las muestras que no participaron en el entrenamiento de cada árbol bootstrap para medir el rendimiento de generalización sin sesgos.

3. **Evaluación de Rendimiento Clínico:** Analizar la matriz de confusión resultante y la capacidad predictiva del modelo frente al severo desbalance de clases del dataset (77.85% Normal vs. 8.28% Patológico), priorizando la detección precisa de riesgos de hipoxia o sufrimiento fetal.

#### **Importación de modelos y preparación de datos**

In [118]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd

# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importamos las clases de los modelos y métricas (están en src)
from src.DecisionTreeClassifier import DecisionTreeClassifier
from src.RandomForest import RandomForestClassifier
from src.Metrics import *

In [119]:
# Carga del dataset limpio de cardiotocografía fetal
clean_data_path = '../data/processed/CTG_clean.csv'
df = pd.read_csv(clean_data_path)

# Separación de la matriz de características 
X = df.drop(columns=["NSP"]).values
y = df["NSP"].values

# Visualización de dataset y dimensiones
display(df.head())
df.info()

print(f"Dimensión de Matriz X (Características): {X.shape}")
print(f"Dimensión de Vector y (Objetivo NSP): {y.shape}")
print(f"Clases en y: {np.unique(y)}")

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006380,0.0,0.006380,0.003190,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003322,0.0,0.008306,0.003322,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.002561,0.0,0.007682,0.002561,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.006515,0.0,0.008143,0.000000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


<class 'pandas.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   LB        2126 non-null   float64
 1   AC        2126 non-null   float64
 2   FM        2126 non-null   float64
 3   UC        2126 non-null   float64
 4   DL        2126 non-null   float64
 5   DS        2126 non-null   float64
 6   DP        2126 non-null   float64
 7   ASTV      2126 non-null   float64
 8   MSTV      2126 non-null   float64
 9   ALTV      2126 non-null   float64
 10  MLTV      2126 non-null   float64
 11  Width     2126 non-null   float64
 12  Min       2126 non-null   float64
 13  Max       2126 non-null   float64
 14  Nmax      2126 non-null   float64
 15  Nzeros    2126 non-null   float64
 16  Mode      2126 non-null   float64
 17  Mean      2126 non-null   float64
 18  Median    2126 non-null   float64
 19  Variance  2126 non-null   float64
 20  Tendency  2126 non-null   float64
 21  NS

In [120]:

# Partición de datos (80% entrenamiento, 20% prueba)
rng = np.random.default_rng(seed=42)
idx_mezcla = rng.permutation(len(X))

n_total = len(X)
idx_train_end = int(0.80 * n_total)

# Dividimos los índices en dos grupos 
train_idx = idx_mezcla[:idx_train_end]
test_idx = idx_mezcla[idx_train_end:]

# Creamos los X y y para cada grupo
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

#### **Entrenamiento de (`RandomForestClassifier`)**

In [121]:
# Entrenamiento del modelo
n_trees = 100 #Numero de árboles
max_depth = 12 #Profundidad máxima de cada rama
min_samples_split= 2 #Mínimo de muestras para dividir un nodo

forest = RandomForestClassifier(n_trees=n_trees, max_depth=max_depth, min_samples_split=min_samples_split, seed=0)
forest.fit(X_train, y_train)

# Cálculo de la precisión Out-Of-Bag (validación interna sin sesgos)
oob = forest.oob_score(X_train, y_train)

# Cálculo de la precisión global sobre el conjunto de entrenamiento
train_preds = forest.predict(X_train)
train_accurracy = (train_preds == y_train).mean()

print("\n--Resultados de Rendimiento en Entrenamiento--")
print(f"Out-Of-Bag (OOB Score): {oob * 100:.2f}%")
print(f"Precisión Global en Entrenamiento:  {train_accurracy * 100:.2f}%")



--Resultados de Rendimiento en Entrenamiento--
Out-Of-Bag (OOB Score): 94.94%
Precisión Global en Entrenamiento:  99.82%


#### **Prueba de (`RandomForestClassifier`)**

In [122]:
#Métricas para validación(Recall y F1-Score)
test_preds = forest.predict(X_test)
test_accurracy = (test_preds == y_test).mean()

#Datos reales y predichos (cambiandolos a positivos e iniciando en 0)
clases_unicas = np.unique(y_test)
y_real_test = (y_test - clases_unicas[0]).astype(int)
y_pred_test = (val_preds - clases_unicas[0]).astype(int)
n_classes = forest.n_classes_

matriz = matrizConfusion(y_real_test, y_pred_test, n_classes)
recalls = recall(matriz)
f1s = f1_score(matriz)


print("\n--Resultado de Rendimiento en Validación--")
print(f"Precisión Global en Validación:  {test_accurracy * 100:.2f}%")

print("\n--Métricas detalladas por Clase--")
nombres_clases = ["1 (Normal)", "2 (Sospechoso)", "3 (Patológico)"]
for i, nombre in enumerate(nombres_clases):
    print(
        f"Clase {nombre} Recall: {recalls[i]*100:.2f}% | F1-Score: {f1s[i]*100:.2f}%"
    )
print(
    f"\nPromedio F1-Score: {np.mean(f1s)*100:.2f}%"
)  



--Resultado de Rendimiento en Validación--
Precisión Global en Validación:  93.66%

--Métricas detalladas por Clase--
Clase 1 (Normal) Recall: 98.25% | F1-Score: 97.40%
Clase 2 (Sospechoso) Recall: 83.02% | F1-Score: 84.62%
Clase 3 (Patológico) Recall: 82.61% | F1-Score: 88.37%

Promedio F1-Score: 90.13%


#### Análisis y Conclusiones finales del Modelo

Durante el desarrollo y entrenamiento de nuestro modelo de Random Forest la primera barrera fue el gran desbalance de clases en los datos. Al analizar los registros, notamos que el 77.85% de los pacientes corresponden a fetos en estado Normal (Clase 1), el 13.88% a estado Sospechoso (Clase 2) y apenas el 8.28% a casos Patológicos (Clase 3). En nuestras primeras pruebas, cuando entrenamos el bosque con un muestreo aleatorio normal y una partición 80% entrenamiento, 17% prueba y 3% validación, el modelo obtuvo una precisión global de más del 90%. A simple vista parecía un resultado muy bueno, pero al revisar el desempeño por clase descubrimos que la gran mayoría de los casos eran bebés sanos, los árboles aprendieron a lo seguro y a predecir casi siempre el caso "Normal". Esto provocó que en la clase Patológica tuviéramos un Recall de apenas 33.33% y un F1-Score de 40.00%, lo que significaba que de cada tres bebés en peligro real de hipoxia, al modelo se le escapaban dos, algo que inaceptable para un caso médico como este.

Para corregir este sesgo, tuvimos que modificar la forma en que el bosque alimenta a sus árboles durante el entrenamiento y cómo evaluamos los datos. Implementamos un muestreo Bootstrap balanceado para que, al construir cada árbol, el algoritmo tomara la misma proporción de pacientes sanos, sospechosos y patológicos. Además, ajustamos la partición de los datos a un esquema de 80% entrenamiento y 20% validación/prueba para contar con una muestra representativa de casos patológicos al evaluar (el 3% que se tenía antes no era lo suficientemente representativo), y configuramos el bosque con 100 árboles, una profundidad máxima de 25 niveles y un mínimo de 2 muestras para dividir cada nodo. Con esto logramos que los árboles fueran lo suficientemente profundos y detallistas como para aislar los patrones clínicos más complejos sin que la clase mayoritaria arrasara contra las minoritarias.

El cambio en los resultados fue enorme y nos dejó ver por qué es tan importante elegir bien las métricas en un problema médico. La precisión global subió a 93.66%, pero lo verdaderamente valioso estuvo en el análisis por clase a través del Recall y el F1-Score. El Recall nos indica qué porcentaje de los casos reales de cada condición logró detectar el modelo; es decir, mide la capacidad para no dejar pasar enfermos por error (evitar falsos negativos),lo que es lo principal y más importante en este problema. Por su parte, el F1-Score funciona para dar un equilibrio porque combina el Recall con la Precisión (qué tantas de nuestras alertas fueron reales y no falsas alarmas), castigando al modelo si falla en cualquiera de los dos lados. Tras los ajustes, la Clase 1 (Normal) alcanzó un Recall de 98.25% y un F1-Score de 97.40%, mientras que la Clase 2 (Sospechoso) subió a un Recall de 83.02% y un F1-Score de 84.62%. Lo más satisfactorio fue que la Clase 3 (Patológico) dio un salto hasta un 82.61% de Recall y un 88.37% de F1-Score, llevando el promedio general de F1-Score del modelo a un sólido 90.13%.

Como conclusión de todo este proceso, aprendimos que en diagnósticos clínicos como el monitoreo fetal nunca debemos confiarnos únicamente de la exactitud o precisión general, ya que el desbalance de datos oculta errores graves en los casos más críticos. El hecho de que en la clase Patológica nuestro F1-Score (88.37%) sea incluso más alto que el Recall (82.61%) demuestra que cuando el modelo emite una alerta de sufrimiento fetal es sumamente confiable y casi no da falsas alarmas, detectando correctamente a más de 8 de cada 10 bebés en riesgo crítico. En resumen, balancear el muestreo interno de los árboles y permitirles crecer con buena profundidad transformó un modelo que solo sabía identificar pacientes sanos en una herramienta equilibrada, sensible y realmente útil para apoyar la detección oportuna de complicaciones fetales.